## Qwen2.5-VL-7B — Public 0.56 모델의 전체 Train split 추가 학습

이 노트북은 첫 제출에 사용한 **단일 Qwen2.5-VL-7B LoRA adapter**를 보존한 채 이어서 학습합니다. 전체 모델 파라미터를 업데이트하는 full fine-tuning이 아니라, 기존 LoRA 파라미터만 추가로 업데이트합니다.

실행 순서는 다음과 같습니다.

1. 현재 adapter와 고정 split을 검사합니다.
2. 현재 adapter를 일반 Validation 200개와 Clean Validation 200개에서 평가합니다.
3. 고정 Train split 7,628개를 모두 한 번씩 사용해 LoRA를 1 epoch 추가 학습합니다.
4. 약 0.5 epoch checkpoint와 최종 adapter의 Validation Exact Match를 비교합니다.
5. Validation으로 선택한 단일 모델만 Clean Validation에서 추가 진단합니다.
6. 결과를 사람이 확인한 뒤에만 마지막 Test 추론 셀을 실행합니다.

Test 데이터는 마지막 제출 추론 전까지 열지 않으며, Test의 분포나 특징을 모델 선택에 사용하지 않습니다.

### 1. A100, Google Drive, 기존 adapter 확인

이 셀은 GPU와 필수 파일 경로를 검사합니다. 첫 제출 모델의 adapter 파일이 실제로 Google Drive에 저장되어 있는지도 확인합니다. 기존 모델을 덮어쓰는 작업은 하지 않습니다.

In [ ]:
import os
import sys
from pathlib import Path

import torch
from google.colab import drive

assert torch.cuda.is_available(), 'GPU runtime이 아닙니다.'
assert torch.cuda.is_bf16_supported(), 'BF16 지원 GPU가 필요합니다.'
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/SNU')
CURRENT_RUN_NAME = 'train4000_augmented_once_1epoch_seed42_v1'
CURRENT_RUN_DIR = (
    PROJECT_DIR / 'outputs' / 'qwen25_vl_7b_lora' / CURRENT_RUN_NAME
)
CURRENT_ADAPTER = CURRENT_RUN_DIR / 'final_adapter'

required = [
    PROJECT_DIR / 'data' / 'train.csv',
    PROJECT_DIR / 'data' / 'train',
    PROJECT_DIR / 'outputs' / 'splits' / 'train_ids.csv',
    PROJECT_DIR / 'outputs' / 'splits' / 'validation_ids.csv',
    PROJECT_DIR / 'outputs' / 'splits' / 'validation_clean_ids.csv',
    PROJECT_DIR / 'scripts' / 'train_qwen_lora.py',
    PROJECT_DIR / 'scripts' / 'evaluate_qwen_baseline.py',
    PROJECT_DIR / 'src' / 'qwen_training.py',
    CURRENT_ADAPTER / 'adapter_model.safetensors',
    CURRENT_ADAPTER / 'adapter_config.json',
]
for path in required:
    assert path.exists(), f'필수 경로를 찾지 못했습니다: {path}'

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

gpu = torch.cuda.get_device_properties(0)
print(f'PyTorch/CUDA: {torch.__version__}/{torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {gpu.total_memory / 1024**3:.1f} GB')
print(f'기존 adapter: {CURRENT_ADAPTER}')
print('기존 adapter 확인 완료: 이 경로는 학습 출력 경로로 사용하지 않습니다.')

### 2. 학습 패키지 설치

검증된 버전의 Transformers와 PEFT를 설치합니다. Colab에 포함된 PyTorch/CUDA는 교체하지 않습니다. 현재 BF16 LoRA에서 사용하지 않고 PEFT와 충돌할 수 있는 구버전 `torchao`는 제거합니다. 설치 후 패키지 import가 이상하면 런타임을 재시작하고 1번 셀부터 다시 실행합니다.

In [ ]:
%pip install -q transformers==5.13.0 accelerate==1.14.0 qwen-vl-utils==0.0.14 peft==0.19.1
%pip uninstall -y torchao

### 3. 학습·제출 관련 테스트

모델을 GPU에 올리기 전에 Answer 순열 변환, completion-only loss, Qwen 출력 parser, submission 행 순서를 검사합니다. 이 테스트는 성능을 평가하는 코드가 아니라 **파이프라인 구현 오류를 잡는 코드**입니다.

In [ ]:
import subprocess

tests_dir = PROJECT_DIR / 'tests'
for pattern in ['test_qwen*.py', 'test_submission.py']:
    result = subprocess.run(
        [
            sys.executable, '-m', 'unittest', 'discover',
            '-s', str(tests_dir), '-p', pattern, '-v',
        ],
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(
            f'{pattern} 테스트 실패 (종료 코드 {result.returncode})'
        )

### 4. 고정 Train/Validation 범위 감사

이번 학습이 원본 Train 9,535개 전체를 무작정 사용하는 것이 아니라, 이미 고정한 **Train 7,628개만 학습**하고 Validation 1,907개는 끝까지 정답 평가용으로 보존하는지 확인합니다. Clean Validation 971개는 Validation 안의 진단용 부분집합이므로 별도의 두 번째 split이 아닙니다.

In [ ]:
from src.qwen_training import load_labeled_split_samples

TRAIN_SPLIT_CSV = PROJECT_DIR / 'outputs' / 'splits' / 'train_ids.csv'
VALIDATION_SPLIT_CSV = PROJECT_DIR / 'outputs' / 'splits' / 'validation_ids.csv'
CLEAN_VALIDATION_SPLIT_CSV = (
    PROJECT_DIR / 'outputs' / 'splits' / 'validation_clean_ids.csv'
)

all_train = load_labeled_split_samples(
    data_dir=PROJECT_DIR / 'data',
    split_csv=TRAIN_SPLIT_CSV,
    expected_split='train',
)
all_validation = load_labeled_split_samples(
    data_dir=PROJECT_DIR / 'data',
    split_csv=VALIDATION_SPLIT_CSV,
    expected_split='validation',
)
clean_validation = load_labeled_split_samples(
    data_dir=PROJECT_DIR / 'data',
    split_csv=CLEAN_VALIDATION_SPLIT_CSV,
    expected_split='validation',
)
assert len(all_train) == 7_628
assert len(all_validation) == 1_907
assert len(clean_validation) == 971
assert {row.sample_id for row in all_train}.isdisjoint(
    {row.sample_id for row in all_validation}
)
assert {row.sample_id for row in clean_validation} <= {
    row.sample_id for row in all_validation
}
print(f'학습에 사용 가능한 고정 Train: {len(all_train):,}')
print(f'학습에서 제외된 고정 Validation: {len(all_validation):,}')
print(f'Clean Validation 부분집합: {len(clean_validation):,}')
print('Train/Validation ID 교집합: 0')

### 5. 공통 실행·평가 함수

긴 subprocess 로그를 Colab 화면에 실시간으로 출력하는 함수와 adapter 평가 함수를 정의합니다. 평가 함수는 정답이 있는 Train Validation만 열며 Test에는 접근하지 않습니다. 이미 완료된 동일 평가가 있으면 결과를 재사용합니다.

In [ ]:
import json

MODEL_NAME = 'Qwen/Qwen2.5-VL-7B-Instruct'
CACHE_DIR = Path('/content/huggingface/hub')

def run_streaming(command, *, cwd):
    process = subprocess.Popen(
        command,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    recent_lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        recent_lines.append(line.rstrip())
        recent_lines = recent_lines[-80:]
    code = process.wait()
    if code != 0:
        raise RuntimeError(
            f'명령 실패 (종료 코드 {code}). 마지막 로그:\n'
            + '\n'.join(recent_lines)
        )

def evaluate_adapter(*, adapter_path, split_csv, output_root, limit=200):
    summary_path = output_root / f'validation_{limit}' / 'summary.json'
    if summary_path.is_file():
        print(f'완료된 평가를 재사용합니다: {summary_path}')
        return summary_path
    command = [
        sys.executable, '-u', '-m', 'scripts.evaluate_qwen_baseline',
        '--data-dir', str(PROJECT_DIR / 'data'),
        '--validation-split-csv', str(split_csv),
        '--output-dir', str(output_root),
        '--model-name', MODEL_NAME,
        '--adapter-path', str(adapter_path),
        '--prompt-version', 'v2_direct_answer',
        '--cache-dir', str(CACHE_DIR),
        '--device', 'cuda',
        '--dtype', 'bfloat16',
        '--limit', str(limit),
        '--seed', '42',
        '--max-new-tokens', '16',
    ]
    run_streaming(command, cwd=PROJECT_DIR)
    assert summary_path.is_file(), f'평가 summary가 없습니다: {summary_path}'
    return summary_path

def read_summary(path):
    return json.loads(path.read_text(encoding='utf-8'))

### 6. 학습 전 현재 모델 — 일반 Validation 200개

첫 제출 모델의 기준 성능을 기존 50개보다 큰 고정 200개에서 다시 측정합니다. 공식 판단값은 네 위치를 모두 맞힌 비율인 `Exact Match Accuracy`입니다. 이 결과는 추가 학습 전 기준선입니다. A100에서 이전 실측 기준 약 10~15분이 예상됩니다.

In [ ]:
BASELINE_EVAL_ROOT = CURRENT_RUN_DIR / 'expanded_validation_before_full_train'
BASELINE_STANDARD_SUMMARY = evaluate_adapter(
    adapter_path=CURRENT_ADAPTER,
    split_csv=VALIDATION_SPLIT_CSV,
    output_root=BASELINE_EVAL_ROOT / 'standard',
    limit=200,
)

### 7. 학습 전 현재 모델 — Clean Validation 200개

Clean Validation은 Train 쪽과 비균일 exact image byte를 공유하지 않는 Validation 부분집합입니다. 일반 Validation 점수와의 차이를 통해 모델이 중복 이미지에 얼마나 의존하는지 진단합니다. 이 점수는 일반 Validation을 대체하지 않습니다.

In [ ]:
BASELINE_CLEAN_SUMMARY = evaluate_adapter(
    adapter_path=CURRENT_ADAPTER,
    split_csv=CLEAN_VALIDATION_SPLIT_CSV,
    output_root=BASELINE_EVAL_ROOT / 'clean',
    limit=200,
)

### 8. 학습 전 기준선 확인

두 Validation 결과의 Exact Match, 위치별 정확도, pairwise ordering 정확도와 출력 파싱 성공률을 한 표로 확인합니다. 파싱 성공률이 100%가 아니면 학습보다 먼저 출력 형식 문제를 해결해야 합니다.

In [ ]:
import pandas as pd
from IPython.display import display

baseline_standard = read_summary(BASELINE_STANDARD_SUMMARY)
baseline_clean = read_summary(BASELINE_CLEAN_SUMMARY)

def summary_row(name, summary):
    return {
        'evaluation': name,
        'sample_count': summary['sample_count'],
        'exact_match': summary['metrics']['exact_match_accuracy'],
        'position_accuracy': summary['metrics']['position_accuracy'],
        'pairwise_accuracy': summary['metrics']['pairwise_ordering_accuracy'],
        'kendall_tau': summary['metrics']['mean_kendall_tau'],
        'parse_success': summary['parsing']['success_rate'],
        'unique_answers': summary['predictions']['unique_answer_count'],
    }

display(pd.DataFrame([
    summary_row('현재 모델 / 일반 Validation', baseline_standard),
    summary_row('현재 모델 / Clean Validation', baseline_clean),
]))
assert baseline_standard['parsing']['success_rate'] == 1.0
assert baseline_clean['parsing']['success_rate'] == 1.0
print('학습 전 기준선 저장 완료. 아직 Test 데이터에는 접근하지 않았습니다.')

### 9. 전체 Train split 학습 예시 감사

7,628개 source ID가 정확히 한 번씩 포함되고, ID마다 이미지 순서를 결정적으로 한 번 재배열한 증강 예시를 만드는지 검사합니다. 이미지 순서는 바뀌어도 Answer가 함께 변환되므로 정답 의미는 유지됩니다.

기존 4,000개와 새 3,628개를 따로 학습하지 않고 7,628개를 함께 섞습니다. 이는 새 데이터만 연속해서 학습할 때 발생할 수 있는 기존 지식 망각을 줄이기 위한 선택입니다.

In [ ]:
from collections import Counter
from src.qwen_training import build_augmented_only_examples

train_examples = build_augmented_only_examples(all_train, seed=42)
label_counts = Counter(row.sample.answer for row in train_examples)
assert len(train_examples) == 7_628
assert len({row.source_sample_id for row in train_examples}) == 7_628
assert all(tuple(sorted(row.sample.answer)) == (1, 2, 3, 4) for row in train_examples)
assert len(label_counts) == 24
effective_batch_size = 8
expected_steps = (len(train_examples) + effective_batch_size - 1) // effective_batch_size
assert expected_steps == 954
print(f'고유 Train source ID: {len(all_train):,}')
print(f'증강 후 학습 예시: {len(train_examples):,}')
print(f'유효 label 종류: {len(label_counts)}')
print(f'label별 예시 수 범위: {min(label_counts.values())}~{max(label_counts.values())}')
print(f'예상 optimizer steps: {expected_steps:,}')
print('이전 실측 속도 기준 학습 예상 시간: 약 5시간 전후')

### 10. Train 7,628개로 LoRA 이어 학습

이 셀이 실제 **모델 fine-tuning 셀**입니다. 현재 adapter에서 이어서 LoRA 파라미터만 업데이트합니다.

- learning rate: `5e-6` — 기존 학습보다 낮춰 급격한 망각과 과적합 위험을 줄입니다.
- epoch: `1` — 7,628개 증강 예시를 한 번씩 봅니다.
- effective batch size: `8` — 실제 batch 1을 8회 누적해 한 번 업데이트합니다.
- checkpoint: 477 step(약 0.5 epoch)과 954 step(약 1 epoch)에 평가·저장합니다.
- training eval loss: Validation 원본 200개로 확인하지만, 최종 모델 선택은 생성 Exact Match로 합니다.

출력 폴더가 이미 있으면 중단하여 기존 결과를 덮어쓰지 않습니다.

In [ ]:
RUN_NAME = 'train7628_augmented_once_continue_lr5e6_1epoch_seed42_v1'
TRAIN_RUN_DIR = PROJECT_DIR / 'outputs' / 'qwen25_vl_7b_lora' / RUN_NAME
assert not TRAIN_RUN_DIR.exists(), (
    f'기존 출력 폴더가 있습니다. 결과 보호를 위해 RUN_NAME을 바꾸세요: {TRAIN_RUN_DIR}'
)
assert CURRENT_ADAPTER.resolve() != (TRAIN_RUN_DIR / 'final_adapter').resolve()

train_command = [
    sys.executable, '-u', '-m', 'scripts.train_qwen_lora',
    '--data-dir', str(PROJECT_DIR / 'data'),
    '--train-split-csv', str(TRAIN_SPLIT_CSV),
    '--validation-split-csv', str(VALIDATION_SPLIT_CSV),
    '--output-dir', str(TRAIN_RUN_DIR),
    '--model-name', MODEL_NAME,
    '--initial-adapter-path', str(CURRENT_ADAPTER),
    '--cache-dir', str(CACHE_DIR),
    '--train-limit', '0',
    '--validation-limit', '200',
    '--augmentation-copies', '1',
    '--augmented-only',
    '--max-steps', '-1',
    '--epochs', '1',
    '--batch-size', '1',
    '--eval-batch-size', '1',
    '--gradient-accumulation-steps', '8',
    '--learning-rate', '5e-6',
    '--logging-steps', '25',
    '--eval-steps', '477',
    '--save-steps', '477',
    '--save-total-limit', '3',
    '--seed', '42',
]
run_streaming(train_command, cwd=PROJECT_DIR)

### 11. 학습 완료 및 loss 기록 확인

최종 adapter가 저장되었는지 확인하고 training loss와 evaluation loss 기록을 표시합니다. Loss가 내려가는 것은 학습이 작동한다는 신호지만, 대회 점수와 동일하지 않으므로 다음 단계에서 반드시 생성 Exact Match를 측정해야 합니다.

In [ ]:
FINAL_ADAPTER = TRAIN_RUN_DIR / 'final_adapter'
MIDPOINT_ADAPTER = TRAIN_RUN_DIR / 'checkpoints' / 'checkpoint-477'
assert (FINAL_ADAPTER / 'adapter_model.safetensors').is_file()
assert (MIDPOINT_ADAPTER / 'adapter_model.safetensors').is_file()

training_summary = read_summary(TRAIN_RUN_DIR / 'training_summary.json')
run_config = read_summary(TRAIN_RUN_DIR / 'run_config.json')
print(json.dumps(training_summary, ensure_ascii=False, indent=2))
print(json.dumps(run_config['split_counts'], ensure_ascii=False, indent=2))

state_path = TRAIN_RUN_DIR / 'checkpoints' / 'trainer_state.json'
if state_path.exists():
    state = read_summary(state_path)
    display(pd.DataFrame(state.get('log_history', [])))

### 12. 0.5 epoch와 1 epoch 모델의 일반 Validation 200개 평가

학습 중간 모델과 최종 모델을 학습 전 기준선과 똑같은 일반 Validation 200개에서 평가합니다. 동일한 ID, prompt, decoding 설정을 사용해야 학습 효과를 공정하게 비교할 수 있습니다. 두 모델 평가에는 이전 실측 기준 총 20~30분 정도가 예상됩니다.

In [ ]:
CANDIDATE_EVAL_ROOT = TRAIN_RUN_DIR / 'candidate_evaluation'
MIDPOINT_STANDARD_SUMMARY = evaluate_adapter(
    adapter_path=MIDPOINT_ADAPTER,
    split_csv=VALIDATION_SPLIT_CSV,
    output_root=CANDIDATE_EVAL_ROOT / 'midpoint_standard',
    limit=200,
)
FINAL_STANDARD_SUMMARY = evaluate_adapter(
    adapter_path=FINAL_ADAPTER,
    split_csv=VALIDATION_SPLIT_CSV,
    output_root=CANDIDATE_EVAL_ROOT / 'final_standard',
    limit=200,
)

### 13. Validation Exact Match로 단일 후보 선택

현재 모델, 0.5 epoch 모델, 1 epoch 모델을 비교합니다. 가장 높은 일반 Validation Exact Match를 우선하고, 동률이면 position accuracy와 pairwise ordering accuracy 순으로 선택합니다. Test 결과를 보면서 모델을 고르지 않습니다.

In [ ]:
candidate_specs = [
    ('before_full_train', CURRENT_ADAPTER, BASELINE_STANDARD_SUMMARY),
    ('half_epoch_step477', MIDPOINT_ADAPTER, MIDPOINT_STANDARD_SUMMARY),
    ('full_epoch_final', FINAL_ADAPTER, FINAL_STANDARD_SUMMARY),
]
candidate_rows = []
for name, adapter_path, summary_path in candidate_specs:
    summary = read_summary(summary_path)
    row = summary_row(name, summary)
    row['adapter_path'] = str(adapter_path)
    candidate_rows.append(row)

candidate_table = pd.DataFrame(candidate_rows).sort_values(
    ['exact_match', 'position_accuracy', 'pairwise_accuracy'],
    ascending=False,
)
display(candidate_table)
best_row = candidate_table.iloc[0]
BEST_NAME = str(best_row['evaluation'])
BEST_ADAPTER = Path(str(best_row['adapter_path']))
BEST_STANDARD_SUMMARY = next(
    summary_path
    for name, _, summary_path in candidate_specs
    if name == BEST_NAME
)
selection_record = {
    'selection_scope': 'train_validation_only_no_test_access',
    'selection_metric': 'exact_match_then_position_then_pairwise',
    'selected_name': BEST_NAME,
    'selected_adapter': str(BEST_ADAPTER),
    'standard_validation_summary': str(BEST_STANDARD_SUMMARY),
    'candidates': candidate_rows,
}
selection_path = TRAIN_RUN_DIR / 'selected_candidate.json'
selection_path.write_text(
    json.dumps(selection_record, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
top_margin = float(candidate_table.iloc[0]['exact_match']) - float(
    candidate_table.iloc[1]['exact_match']
)
print(f'선택된 단일 모델: {BEST_NAME}')
print(f'adapter: {BEST_ADAPTER}')
print(f'1위와 2위 Exact Match 차이: {top_margin:+.4f}')
print(f'선택 기록: {selection_path}')
if top_margin < 0.02:
    print('주의: 200개 기준 차이가 2%p 미만입니다. Test 전 평가 확대를 검토하세요.')

### 14. 선택 모델의 Clean Validation 200개 진단

일반 Validation으로 선택한 모델을 Clean Validation에서도 평가합니다. 일반 점수보다 Clean 점수가 크게 낮다면 정확한 중복이 없는 샘플에서 일반화가 약하다는 뜻입니다. 모델 선택을 Clean 점수에 맞춰 반복하면서 Test를 탐색하지는 않습니다.

In [ ]:
if BEST_NAME == 'before_full_train':
    BEST_CLEAN_SUMMARY = BASELINE_CLEAN_SUMMARY
else:
    BEST_CLEAN_SUMMARY = evaluate_adapter(
        adapter_path=BEST_ADAPTER,
        split_csv=CLEAN_VALIDATION_SPLIT_CSV,
        output_root=CANDIDATE_EVAL_ROOT / f'{BEST_NAME}_clean',
        limit=200,
    )
best_standard = read_summary(BEST_STANDARD_SUMMARY)
best_clean = read_summary(BEST_CLEAN_SUMMARY)
display(pd.DataFrame([
    summary_row(f'{BEST_NAME} / 일반 Validation', best_standard),
    summary_row(f'{BEST_NAME} / Clean Validation', best_clean),
]))
assert best_standard['parsing']['success_rate'] == 1.0
assert best_clean['parsing']['success_rate'] == 1.0
gap = (
    best_standard['metrics']['exact_match_accuracy']
    - best_clean['metrics']['exact_match_accuracy']
)
print(f'일반 - Clean Exact Match 차이: {gap:+.4f}')
print('여기까지는 Train에서 분리한 Validation만 사용했습니다.')

### 15. 선택 모델로 Test 819개 추론 — 결과 확인 후에만 실행

이 셀부터 처음으로 Test 이미지를 엽니다. Test 정답 분포나 예측 분포를 분석하지 않고, Validation에서 이미 선택된 단일 adapter로 819개를 순서대로 추론합니다. 모든 출력이 유효한 1~4 순열로 파싱된 경우에만 `submission.csv`를 생성합니다.

**14번 셀의 성능 표를 확인하기 전에는 이 셀을 실행하지 마세요.**

In [ ]:
import csv
from src.submission import validate_submission_file

test_required = [
    PROJECT_DIR / 'data' / 'test.csv',
    PROJECT_DIR / 'data' / 'test',
    PROJECT_DIR / 'data' / 'sample_submission.csv',
    PROJECT_DIR / 'scripts' / 'predict_qwen_test.py',
]
for path in test_required:
    assert path.exists(), f'Test 추론 필수 경로가 없습니다: {path}'

safe_best_name = BEST_NAME.replace('/', '_').replace(' ', '_')
TEST_OUTPUT = TRAIN_RUN_DIR / f'test_submission_{safe_best_name}'
submission_path = TEST_OUTPUT / 'submission.csv'
if not submission_path.is_file():
    test_command = [
        sys.executable, '-u', '-m', 'scripts.predict_qwen_test',
        '--data-dir', str(PROJECT_DIR / 'data'),
        '--output-dir', str(TEST_OUTPUT),
        '--model-name', MODEL_NAME,
        '--adapter-path', str(BEST_ADAPTER),
        '--cache-dir', str(CACHE_DIR),
        '--device', 'cuda',
        '--dtype', 'bfloat16',
        '--max-new-tokens', '16',
    ]
    run_streaming(test_command, cwd=PROJECT_DIR)
else:
    print(f'기존 제출 파일을 재사용합니다: {submission_path}')

with (PROJECT_DIR / 'data' / 'sample_submission.csv').open(
    newline='', encoding='utf-8-sig'
) as handle:
    template_ids = [row['Id'] for row in csv.DictReader(handle)]
validated_rows = validate_submission_file(submission_path, template_ids)
inference_summary = read_summary(TEST_OUTPUT / 'inference_summary.json')
assert validated_rows == 819
assert inference_summary['parse_failure_count'] == 0
print(f'제출 파일: {submission_path}')
print(f'검증된 행: {validated_rows:,}')
print('열: Id, Answer')
print('파싱 실패: 0')

### 실행 후 보존해야 할 결과

새 실행 폴더에는 다음 결과가 저장됩니다.

- `run_config.json`: 학습 데이터 범위와 hyperparameter
- `training_summary.json`: 학습·검증 loss와 실행 시간
- `checkpoints/checkpoint-477/`: 약 0.5 epoch adapter
- `final_adapter/`: 1 epoch 최종 adapter
- `candidate_evaluation/`: checkpoint별 Validation 예측과 성능
- `selected_candidate.json`: Test를 보기 전에 선택한 단일 모델 기록
- `test_submission_<선택 모델>/submission.csv`: 구조 검증을 통과한 제출 파일

첫 제출에 사용한 `train4000_augmented_once_1epoch_seed42_v1/final_adapter`는 변경되지 않습니다.